In [1]:
import sys
sys.path.append("../..")

from animgen.core.models.model import BaseModelClass

import numpy as np
from pathlib import Path
import trimesh

In [6]:
SEGMENTED_PATHS = {
    "tail": Path("../../generated_data/test_segmented/tail.glb"),
    "side_fins": Path("../../generated_data/test_segmented/side_fins.glb"),
    "top_fins": Path("../../generated_data/test_segmented/top_fins.glb"),
    "body": Path("../../generated_data/test_segmented/body.glb"),
    "snake_body": Path("../../generated_data/test_segmented/snake_body_stright_test.glb"),
    "snake_complete": Path("../../generated_data/models/paint_mesh_Sea_Snake.glb"),
}

In [7]:
mesh_snake_complete = BaseModelClass(SEGMENTED_PATHS["snake_complete"]).mesh
mesh_snake_body = BaseModelClass(SEGMENTED_PATHS["snake_body"]).mesh

Rendering Multiviews...: 100%|██████████| 20/20 [00:04<00:00,  4.43it/s]


In [8]:
def sample_points_from_mesh(mesh, num_points=1000):
    """
    Uniformly sample vertex indices from a mesh.

    Parameters
    ----------
    mesh : trimesh.Trimesh
        Input mesh.

    num_points : int
        Number of vertices to sample.

    Returns
    -------
    sampled_indices : np.ndarray
        Shape (num_points,) containing sampled vertex indices.
    """
    num_points = min(num_points, len(mesh.vertices))

    sampled_indices = np.random.choice(
        len(mesh.vertices),
        size=num_points,
        replace=False
    )

    return sampled_indices

In [9]:
def batched_PCA(points):
    """
    Perform batched PCA on 3D point clouds.

    Parameters
    ----------
    points : np.ndarray
        Shape (B, N, 3), where:
        B = batch size
        N = number of points

    Returns
    -------
    pca_components : np.ndarray
        Shape (B, 3, 3).

    eigenvalues : np.ndarray
        Shape (B, 3), sorted largest -> smallest.
    """

    # (B, N, 3)
    centered_points = points - np.mean(
        points,
        axis=1,
        keepdims=True
    )

    cov_matrix = np.einsum(
        "bni,bnj->bij",
        centered_points,
        centered_points
    ) / (points.shape[1] - 1)

    eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)

    eigenvalues = eigenvalues[:, ::-1]
    eigenvectors = eigenvectors[:, :, ::-1]

    return eigenvectors, eigenvalues

In [10]:
def find_k_neighbours(mesh: trimesh.Trimesh, point_idx: int, k: int = 2):
    """
    Find all vertices within k connectivity rings of a given mesh vertex.

    Parameters
    ----------
    mesh : trimesh.Trimesh
        Input triangular mesh.

    point_idx : int
        Index of the center vertex.

    k : int, default=2
        Number of connectivity rings to traverse.

    Returns
    -------
    neighbours : np.ndarray
        Array of shape (N, 3) containing the center vertex and all
        vertices reachable within k mesh-edge hops.
    """

    visited = {point_idx}

    frontier = {point_idx}

    for _ in range(k):
        next_frontier = set()

        for vertex_idx in frontier:
            next_frontier.update(mesh.vertex_neighbors[vertex_idx])

        next_frontier -= visited

        visited.update(next_frontier)
        frontier = next_frontier

        if not frontier:
            break

    indices = np.fromiter(visited, dtype=np.int64)

    neighbours = mesh.vertices[indices]

    return neighbours

In [11]:
def structural_descriptors(eigenvalues):
    """
    Calculate PCA-based structural descriptors from eigenvalues.

    Parameters
    ----------
    eigenvalues : array-like
        Shape (3,) for a single PCA or (B, 3) for batched PCA.
        Eigenvalues do not need to be pre-sorted.

    Returns
    -------
    dict
        For single input:
            linearity, planarity, sphericity

        For batched input:
            linearity, planarity, sphericity : shape (B,)
            mean : shape (3,)
            sum  : shape (3,)
            std  : shape (3,)
    """
    eigenvalues = np.asarray(eigenvalues, dtype=float)

    # Single PCA
    if eigenvalues.ndim == 1:
        lambda1, lambda2, lambda3 = np.sort(eigenvalues)[::-1]

        if lambda1 <= 0:
            raise ValueError("Largest eigenvalue must be positive.")

        linearity = (lambda1 - lambda2) / lambda1
        planarity = (lambda2 - lambda3) / lambda1
        sphericity = lambda3 / lambda1

        return {
            "linearity": linearity,
            "planarity": planarity,
            "sphericity": sphericity,
        }

    # Batched PCA: (B, 3)
    if eigenvalues.ndim == 2:
        eigenvalues = np.sort(eigenvalues, axis=1)[:, ::-1]

        lambda1 = eigenvalues[:, 0]
        lambda2 = eigenvalues[:, 1]
        lambda3 = eigenvalues[:, 2]

        if np.any(lambda1 <= 0):
            raise ValueError("Largest eigenvalue must be positive.")

        linearity = (lambda1 - lambda2) / lambda1
        planarity = (lambda2 - lambda3) / lambda1
        sphericity = lambda3 / lambda1

        descriptors = np.stack(
            [linearity, planarity, sphericity],
            axis=1
        )  # (B, 3)

        return {
            "linearity": linearity,
            "planarity": planarity,
            "sphericity": sphericity,

            "mean": np.mean(descriptors, axis=0),
            "sum": np.sum(descriptors, axis=0),
            "std": np.std(descriptors, axis=0),
        }

    raise ValueError(
        "Eigenvalues must have shape (3,) or (B, 3)."
    )

In [12]:
K = 3
NUM_SAMPLES = 1000


# ============================================================
# Straight Snake
# ============================================================

print("Snake Straight Body structural descriptors:")

sampled_indices = sample_points_from_mesh(
    mesh_snake_body,
    num_points=NUM_SAMPLES
)

snake_body_eigenvalues = []

for point_idx in sampled_indices:

    neighbours = find_k_neighbours(
        mesh_snake_body,
        point_idx,
        k=K
    )

    _, eigenvalues = batched_PCA(neighbours[None, ...])

    snake_body_eigenvalues.append(eigenvalues[0])


snake_body_eigenvalues = np.stack(
    snake_body_eigenvalues,
    axis=0
)

snake_body_desc = structural_descriptors(
    snake_body_eigenvalues
)


print("Mean:")
print(f"  Linearity:  {snake_body_desc['mean'][0]}")
print(f"  Planarity:  {snake_body_desc['mean'][1]}")
print(f"  Sphericity: {snake_body_desc['mean'][2]}")

print("Std:")
print(f"  Linearity:  {snake_body_desc['std'][0]}")
print(f"  Planarity:  {snake_body_desc['std'][1]}")
print(f"  Sphericity: {snake_body_desc['std'][2]}")


# ============================================================
# Curved Snake
# ============================================================

print("\nSnake Curved structural descriptors:")

sampled_indices = sample_points_from_mesh(
    mesh_snake_complete,
    num_points=NUM_SAMPLES
)

snake_curved_eigenvalues = []

for point_idx in sampled_indices:

    neighbours = find_k_neighbours(
        mesh_snake_complete,
        point_idx,
        k=K
    )

    _, eigenvalues = batched_PCA(neighbours[None, ...])

    snake_curved_eigenvalues.append(eigenvalues[0])


# (NUM_SAMPLES, 3)
snake_curved_eigenvalues = np.stack(
    snake_curved_eigenvalues,
    axis=0
)

snake_curved_desc = structural_descriptors(
    snake_curved_eigenvalues
)


print("Mean:")
print(f"  Linearity:  {snake_curved_desc['mean'][0]}")
print(f"  Planarity:  {snake_curved_desc['mean'][1]}")
print(f"  Sphericity: {snake_curved_desc['mean'][2]}")

print("Std:")
print(f"  Linearity:  {snake_curved_desc['std'][0]}")
print(f"  Planarity:  {snake_curved_desc['std'][1]}")
print(f"  Sphericity: {snake_curved_desc['std'][2]}")

Snake Straight Body structural descriptors:
Mean:
  Linearity:  0.9783907425961804
  Planarity:  0.021609257403818916
  Sphericity: 1.1364948141124145e-16
Std:
  Linearity:  0.12495327615271754
  Planarity:  0.12495327615271755
  Sphericity: 1.9810019722182451e-16

Snake Curved structural descriptors:
Mean:
  Linearity:  0.874164327152682
  Planarity:  0.12003867730891445
  Sphericity: 0.005796995538403853
Std:
  Linearity:  0.12411970717856942
  Planarity:  0.11947592594932865
  Sphericity: 0.009319457046218295


In [13]:
""" 
Notes
-----
As seen here local PCA is much better as compared to global PCA. The local PCA is able to capture 
the local structure of the mesh, while the global PCA is not able to capture the local structure of the mesh. 

Thus this is much more suitable to work as a classifier in types of meshes.
"""

' \nNotes\n-----\nAs seen here local PCA is much better as compared to global PCA. The local PCA is able to capture \nthe local structure of the mesh, while the global PCA is not able to capture the local structure of the mesh. \n\nThus this is much more suitable to work as a classifier in types of meshes.\n'

In [14]:
K = 1
NUM_SAMPLES = 1000

print("\n Top Fin Discriptors:")

mesh_top_fins = BaseModelClass(SEGMENTED_PATHS["top_fins"]).mesh

sampled_indices = sample_points_from_mesh(
    mesh_top_fins,
    num_points=NUM_SAMPLES
)

mesh_top_fins_eigenvalues = []

for point_idx in sampled_indices:

    neighbours = find_k_neighbours(
        mesh_top_fins,
        point_idx,
        k=K
    )

    _, eigenvalues = batched_PCA(neighbours[None, ...])

    mesh_top_fins_eigenvalues.append(eigenvalues[0])


# (NUM_SAMPLES, 3)
mesh_top_fins_eigenvalues = np.stack(
    mesh_top_fins_eigenvalues,
    axis=0
)

mesh_top_fins_desc = structural_descriptors(
    mesh_top_fins_eigenvalues
)


print("Mean:")
print(f"  Linearity:  {mesh_top_fins_desc['mean'][0]}")
print(f"  Planarity:  {mesh_top_fins_desc['mean'][1]}")
print(f"  Sphericity: {mesh_top_fins_desc['mean'][2]}")

print("Std:")
print(f"  Linearity:  {mesh_top_fins_desc['std'][0]}")
print(f"  Planarity:  {mesh_top_fins_desc['std'][1]}")
print(f"  Sphericity: {mesh_top_fins_desc['std'][2]}")


 Top Fin Discriptors:


Rendering Multiviews...: 100%|██████████| 20/20 [00:04<00:00,  4.05it/s]

Mean:
  Linearity:  0.7961709446472365
  Planarity:  0.20382905535275558
  Sphericity: 8.191903762845444e-15
Std:
  Linearity:  0.17053101434140305
  Planarity:  0.17053101434140527
  Sphericity: 2.5906620232508596e-13


In [15]:
mesh_top_fins.show()

In [ ]:
"""
NOTES:

Again this seems to fail too for planer fins, I guess the reason could be that near the ends the points are wayy too more present,
and thus the PCA is not able to capture the local structure of the mesh. Still it does prove the 
local analysis is still better than global PCA results,

Bias understood here:
- Random Sampling of points is not uniform, for points near the ends are more than in the middle 
- Doing rings around the points is not uniform, as for example in the fin triangles are elongated to one side due to shape

One possible solution is to convert to uniform point cloud, and I am currently working on that direction.
"""